In [2]:
import ee
import geemap

def alphaearth_bands(d: int = 64):
    return [f"A{i:02d}" for i in range(d)]

def get_alphaearth_image_by_year(year: int = 2024, d: int = 64) -> ee.Image:
    dataset = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
    img = dataset.filterDate(f"{year}-01-01", f"{year+1}-01-01").mosaic()
    return img.select(alphaearth_bands(d))

def sample_embedding_feature(img: ee.Image, lon: float, lat: float, scale: int = 10) -> ee.Feature:
    pt = ee.Geometry.Point([lon, lat])
    fc = img.sample(region=pt, scale=scale, numPixels=1, geometries=False)
    return ee.Feature(fc.first())  # may be null server-side if no coverage

def feature_to_constant_embedding_image(f: ee.Feature, bands) -> ee.Image:
    # Make a constant 64-band image whose band values are from the feature
    vec_list = ee.List(bands).map(lambda b: ee.Number(f.get(ee.String(b))))
    return ee.Image.constant(vec_list).rename(bands)

def build_similarity_image(
    img: ee.Image,
    ref_feature: ee.Feature,
    bands,
    metric: str = "cosine",
    eps: float = 1e-12,
) -> ee.Image:
    """
    Returns an ee.Image with one band 'sim' containing dot or cosine similarity
    between each pixel embedding and the reference embedding.
    """
    ref_img = feature_to_constant_embedding_image(ref_feature, bands)

    # dot(x) = sum_i A_i(x) * ref_i
    dot = img.multiply(ref_img).reduce(ee.Reducer.sum())

    if metric == "dot":
        return dot.rename("sim")

    # cosine = dot / (||x|| * ||ref||)
    norm_img = img.multiply(img).reduce(ee.Reducer.sum()).sqrt()
    ref_norm = ref_img.multiply(ref_img).reduce(ee.Reducer.sum()).sqrt()

    denom = norm_img.multiply(ref_norm).max(eps)
    cos = dot.divide(denom)
    return cos.rename("sim")

def alphaearth_similarity_heatmap_map(
    project_id: str,
    lon: float,
    lat: float,
    year: int = 2024,
    scale: int = 10,
    radius_m: float = 2000,
    metric: str = "cosine",   # "cosine" or "dot"
    zoom: int = 14,
):
    """
    Creates and returns a geemap.Map with:
      - reference point marker
      - buffer circle
      - similarity heatmap layer
    """
    # Init EE (you can replace this with your utils.gee_init)
    ee.Authenticate()
    ee.Initialize(project=project_id)

    bands = alphaearth_bands(64)
    img = get_alphaearth_image_by_year(year=year)

    pt = ee.Geometry.Point([lon, lat])
    region = pt.buffer(radius_m)

    # Sample ref embedding
    ref_f = sample_embedding_feature(img, lon, lat, scale=scale)

    # Check coverage by verifying A00 exists & not null
    # We do this server-side using an ee.Algorithms.If
    a00 = ref_f.get("A00")
    is_null = ee.Algorithms.IsEqual(a00, None)

    # Build sim image (only meaningful if not null); still build it,
    # but we’ll warn client-side if null.
    sim_img = build_similarity_image(img, ref_f, bands, metric=metric).clip(region)

    # Create map
    m = geemap.Map()
    m.set_center(lon, lat, zoom)

    # Add marker / buffer
    m.addLayer(pt, {"color": "cyan"}, "Reference point")
    m.addLayer(region, {"color": "cyan"}, f"Buffer {int(radius_m)}m", False)

    # Visualization palette (same vibe as your JS script)
    vis = {
        "min": -0.2,
        "max": 1.0,
        "palette": [
            "000004","1b0c41","4a0c6b","781c6d","a52c60",
            "cf4446","ed6925","fb9b06","f7d13d","fcffa4"
        ],
    }

    m.addLayer(sim_img, vis, f"Similarity heatmap ({metric})")

    # legend_dict = {
    #     f"{vis['min']:.1f} (low)": vis["palette"][0],
    #     f"{(vis['min'] + vis['max']) / 2:.2f}": vis["palette"][len(vis["palette"]) // 2],
    #     f"{vis['max']:.1f} (high)": vis["palette"][-1],
    # }

    PALETTE = [
            "000004","1b0c41","4a0c6b","781c6d","a52c60",
            "cf4446","ed6925","fb9b06","f7d13d","fcffa4"]

    legend_dict = {
        "-0.2": PALETTE[0],
        "0.0": PALETTE[2],
        "0.3": PALETTE[4],
        "0.6": PALETTE[6],
        "0.8": PALETTE[8],
        "1.0": PALETTE[-1],
    }

    m.add_legend(
        title=f"AlphaEarth similarity\n({metric})",
        legend_dict=legend_dict,
        position="bottomright"
    )

    # Print a quick coverage check + (optional) embedding dict
    null_flag = is_null.getInfo()
    if null_flag:
        print("No AlphaEarth embedding coverage at this point (masked / no data). Try nearby land.")
    else:
        # If you want to inspect the reference embedding:
        ref_dict = ref_f.toDictionary(bands).getInfo()
        print(f"Reference embedding at lon={lon}, lat={lat} (year={year}):")
        print(ref_dict)

    return m


if __name__ == "__main__":
    PROJECT_ID = "iconic-smoke-480408-r8"
    lon, lat = 103.8198, 1.3521  # Singapore

    m = alphaearth_similarity_heatmap_map(
        project_id=PROJECT_ID,
        lon=lon,
        lat=lat,
        year=2024,
        scale=10,
        radius_m=3000,
        metric="cosine",
        zoom=14,
    )

Reference embedding at lon=103.8198, lat=1.3521 (year=2024):
{'A00': 0.0271280276816609, 'A01': -0.07972318339100345, 'A02': -0.07972318339100345, 'A03': -0.006151480199923107, 'A04': -0.07111111111111111, 'A05': -0.14769703960015382, 'A06': -0.03844675124951941, 'A07': 0.14173010380622836, 'A08': -0.05536332179930796, 'A09': 0.16000000000000003, 'A10': 0.029773164167627833, 'A11': 0.2069357939254133, 'A12': -0.05911572472126105, 'A13': -0.01384083044982699, 'A14': -0.13016532103037293, 'A15': -0.21413302575932336, 'A16': -0.019930795847750864, 'A17': 0.05173394848135333, 'A18': -0.1085121107266436, 'A19': -0.008858131487889272, 'A20': 0.05911572472126105, 'A21': 0.05173394848135333, 'A22': 0.2364628988850442, 'A23': 0.29287197231833906, 'A24': 0.044844290657439445, 'A25': 0.10340638216070744, 'A26': 0.14769703960015382, 'A27': -0.17279507881584008, 'A28': 0.05173394848135333, 'A29': -0.05536332179930796, 'A30': 0.2069357939254133, 'A31': -0.13588619761630144, 'A32': -0.244152249134948

In [3]:
m

Map(center=[1.3521, 103.8198], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…